# DubFlow - Google Colab AI service

Run all cells. This notebook serves the whole AI layer: speech recognition,
translation, speech generation, speaker diarization and source separation, each
behind a provider that the API reports through `GET /capabilities`. The last
cell prints the values the local application's `.env` needs.

**The base install covers**: every Whisper checkpoint, SeamlessM4T and MMS
recognition, both translation engines, and the `mms` voice. Everything else is
optional - turn it on in the next cell before running the rest.

**Dependency warnings.** These are not interchangeable flags; some of them fight
each other, and a Colab runtime that has to be restarted is the usual symptom.

| Flag | Installs | Known conflicts |
|---|---|---|
| `INSTALL_EDGE` | `edge-tts` | none; safe with everything |
| `INSTALL_PIPER` | `piper-tts` | pins `onnxruntime`; safe in practice |
| `INSTALL_COQUI` | `coqui-tts` | pins `transformers` and `torch`; **do not enable with `INSTALL_F5` or `INSTALL_PARAKEET`** |
| `INSTALL_F5` | `f5-tts` | pins `torch`/`torchaudio`; conflicts with `INSTALL_COQUI` |
| `INSTALL_CHATTERBOX` | `chatterbox-tts` | pins `transformers`; keep it alone with the base install |
| `INSTALL_KOKORO` | `kokoro`, `espeak-ng` | light; Japanese and Chinese need `misaki[ja]` / `misaki[zh]` |
| `INSTALL_OPENVOICE` | OpenVoice V2 + MeloTTS from git | pulls `unidic`, several pinned deps; slow and the most fragile |
| `INSTALL_COSYVOICE` | git clone, not pip | needs `COSYVOICE_ROOT`; heavy |
| `INSTALL_VIENEU` | `vieneu` | pins its own `transformers`; keep it alone |
| `INSTALL_SENSEVOICE` | `funasr` | pins `torch`; keep it alone |
| `INSTALL_PARAKEET` | `nemo_toolkit[asr]` | large, pins `transformers`; **do not combine with `INSTALL_COQUI`** |
| `INSTALL_DIARIZATION` | `pyannote.audio` | needs `HF_TOKEN` and the model conditions accepted |
| `INSTALL_DEMUCS` | `demucs` | safe with the base install |

A session that answers `available: false` for an engine is telling you its
package is missing, not that the engine is broken. Nothing is preloaded: the
first request for a model downloads it, and switching models frees the previous
one.

This server also runs the whole pipeline on its own: `POST /jobs` with a video,
poll `GET /jobs/{id}`, then fetch `GET /jobs/{id}/download`. Nothing has to run
on your machine for that route, but Colab storage is ephemeral - download
results before the session ends.

In [ ]:
REPO_URL = "https://github.com/huynhphatloi/MultilingualVideoDubbingSystem.git"
BRANCH = "main"
PORT = 8000

# Fallback only. Every request may name its own model.
WHISPER_MODEL = "small"

# Gated weights. pyannote needs a Hugging Face token whose account has accepted
# the conditions on pyannote/speaker-diarization-3.1 and pyannote/segmentation-3.0.
HF_TOKEN = ""

# --- Optional speech engines. 'mms' always works without any of these. -----
INSTALL_EDGE = True          # edge        - most natural, no GPU, needs internet
INSTALL_PIPER = False        # piper       - fast local voices
INSTALL_COQUI = False        # xtts_v2, vixtts - cloning; conflicts with F5
INSTALL_F5 = False           # f5_vi, f5_base  - cloning; conflicts with Coqui
INSTALL_CHATTERBOX = False   # chatterbox  - MIT cloning, 23 languages
INSTALL_KOKORO = False       # kokoro      - Apache-2.0, stock voices
INSTALL_OPENVOICE = False    # openvoice_v2 - cloning via MeloTTS; fragile
INSTALL_COSYVOICE = False    # cosyvoice2  - cloning, installed from git
INSTALL_VIENEU = False       # vieneu      - Vietnamese cloning

# --- Optional recognisers. Whisper, Seamless and MMS need nothing extra. ---
INSTALL_SENSEVOICE = False   # sensevoice_small - zh/en/ja/ko
INSTALL_PARAKEET = False     # parakeet_*       - NeMo, large install

# --- Optional pipeline stages --------------------------------------------
INSTALL_DIARIZATION = False  # pyannote.audio - one voice per speaker
INSTALL_DEMUCS = False       # demucs         - separate speech from background
INSTALL_LIPSYNC = False      # no provider ships with this build; see
                             # colab/providers/lipsync/registry.py

In [2]:
from pathlib import Path

if Path("/content/dubflow").exists():
    # Never keep an earlier checkout: both the API contract and the stage code
    # live in this repo, and a stale one fails in ways that look like AI errors.
    !git -C /content/dubflow fetch --depth 1 origin {BRANCH} && git -C /content/dubflow reset --hard FETCH_HEAD
else:
    !git clone --depth 1 --branch {BRANCH} {REPO_URL} /content/dubflow
%cd /content/dubflow/colab
!pip install -q -r requirements.txt

remote: Enumerating objects: 34, done.
remote: Counting objects: 100% (34/34), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 19 (delta 5), reused 15 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (19/19), 24.91 KiB | 439.00 KiB/s, done.
From https://github.com/huynhphatloi/MultilingualVideoDubbingSystem
 * branch            main       -> FETCH_HEAD
 + 628319a...c900c5b main       -> origin/main  (forced update)
HEAD is now at c900c5b feat: update form description and make model fields optional in upload form
/content/dubflow/colab


In [ ]:
# Optional engines. Each one is skipped unless its flag is on above. Enable the
# smallest set you actually intend to test: several of these pin their own torch
# or transformers and will ask Colab to restart.
if INSTALL_EDGE:
    !pip install -q edge-tts
if INSTALL_PIPER:
    !pip install -q piper-tts
if INSTALL_COQUI:
    !pip install -q coqui-tts
if INSTALL_F5:
    !pip install -q f5-tts
if INSTALL_CHATTERBOX:
    !pip install -q chatterbox-tts
if INSTALL_KOKORO:
    !apt-get -qq -y install espeak-ng > /dev/null 2>&1
    !pip install -q kokoro soundfile
if INSTALL_OPENVOICE:
    !pip install -q git+https://github.com/myshell-ai/OpenVoice.git
    !pip install -q git+https://github.com/myshell-ai/MeloTTS.git
    !python -m unidic download
if INSTALL_COSYVOICE:
    # Not on PyPI: the provider reads COSYVOICE_ROOT to find this checkout.
    !test -d /content/CosyVoice || git clone --recursive -q https://github.com/FunAudioLLM/CosyVoice.git /content/CosyVoice
    !pip install -q -r /content/CosyVoice/requirements.txt
    import os; os.environ["COSYVOICE_ROOT"] = "/content/CosyVoice"
if INSTALL_VIENEU:
    !pip install -q vieneu
if INSTALL_SENSEVOICE:
    !pip install -q "funasr>=1.3.26"
if INSTALL_PARAKEET:
    !pip install -q "nemo_toolkit[asr]"
if INSTALL_DIARIZATION:
    !pip install -q "pyannote.audio>=3.1"
if INSTALL_DEMUCS:
    !pip install -q demucs
if INSTALL_LIPSYNC:
    print("No lip-sync provider ships with this build - see "
          "colab/providers/lipsync/registry.py for why and what adding one needs.")

print("Optional packages done. /capabilities lists what this session can load.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.1/34.1 MB 20.5 MB/s eta 0:00:00:00:0100:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 862.8/862.8 kB 5.5 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 997.3/997.3 kB 48.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 639.3/639.3 kB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.5/163.5 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.1/71.1 kB 6.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.9/106.9 kB 5.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 50.5 MB/s eta 0:00:0000:01:00:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.

^C


In [ ]:
import os
import secrets
import subprocess
import threading
import time

import uvicorn

AUTH_TOKEN = secrets.token_urlsafe(24)
os.environ["AUTH_TOKEN"] = AUTH_TOKEN
os.environ["WHISPER_MODEL"] = WHISPER_MODEL
if HF_TOKEN:
    # pyannote reads this; without it diarization reports available: false.
    os.environ["HF_TOKEN"] = HF_TOKEN

import server

# `import` is a no-op once the module is cached, so re-running this cell would
# print a fresh token while the server kept checking the one from the first run
# - every call then comes back "401 bad token". Push the current one in.
server.AUTH_TOKEN = AUTH_TOKEN

if "api_thread" not in globals() or not api_thread.is_alive():
    api_thread = threading.Thread(
        target=lambda: uvicorn.run(server.app, host="0.0.0.0", port=PORT, log_level="warning"),
        daemon=True,
    )
    api_thread.start()
    time.sleep(2)

import providers

print("Colab AI API started. Models load on the first request.")
for task, groups in providers.capabilities()["providers"].items():
    ready = [
        model["id"]
        for group in groups for model in group["models"] if model["available"]
    ]
    print(f"  {task:12} {len(ready):2} ready: {', '.join(ready[:6])}"
          + (" ..." if len(ready) > 6 else ""))

In [ ]:
import re

cloudflared = Path("/content/cloudflared")
if not cloudflared.exists():
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
    !chmod +x /content/cloudflared

if "tunnel" in globals():
    tunnel.terminate()

tunnel = subprocess.Popen(
    [str(cloudflared), "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
public_url = None
for line in iter(tunnel.stdout.readline, ""):
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break

if not public_url:
    raise RuntimeError("Cloudflare tunnel did not return a public URL")

# One line to copy: it writes .env, restarts the API, and reports what is live.
print("Run this in the project directory on your machine:\n")
print(f"  make colab URL={public_url} TOKEN={AUTH_TOKEN}\n")
print("Or set these by hand in .env:")
print(f"  COLAB_API_URL={public_url}")
print(f"  COLAB_API_TOKEN={AUTH_TOKEN}")